## Libraries

In [ ]:
import numpy as np
import pandas as pd
import time
import random

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

from statsmodels.stats.diagnostic import acorr_ljungbox

## Config

In [ ]:
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

# temporal splits
TRAIN_END_DATE  = pd.Timestamp("2021-03-31")   # training period end
VAL_END_DATE    = pd.Timestamp("2022-03-31")   # val period end
TEST_START_DATE = pd.Timestamp("2022-04-01")   # test period start

# === Fill these with your best hyperparameters from tuning ===
WINDOW       = 24     # input length (months)
CNN_CHANNELS = 32     # conv filters
KERNEL_SIZE  = 3      # conv kernel size
LSTM_HIDDEN  = 64     # LSTM hidden units
DROPOUT      = 0.0
LR           = 1e-3
WEIGHT_DECAY = 0.0

BATCH_SIZE    = 64
MAX_EPOCHS    = 80
PATIENCE      = 8
MAX_GRAD_NORM = 5.0

# device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# feature lists
continuous_cols = [
    "AverageNeighbourPrice","local_I","area_km2","centroid_x","centroid_y",
    "CoL_distance_km","LA_FE","sdlt_perc_threshold","dwelling_stock",
    "population","ashe_weekly","base_rate","claimant_count_prop",
    "planning_decisions_per_1000","planning_granted_prop",
    "rail_station_entry_exit","GDP","CPIH"
]

categorical_cols = [
    "LMIQuadrant__2","LMIQuadrant__3","LMIQuadrant__4",
    "Region_East of England","Region_London","Region_North East",
    "Region_North West","Region_South East","Region_South West",
    "Region_West Midlands","Region_Yorkshire and The Humber"
]

base_feature_cols = continuous_cols + categorical_cols

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## Metric functions

In [ ]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(
        100.0 * np.mean(
            2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)
        )
    )

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """
    MASE using seasonal naive of lag m on TRAIN (pre-test) period.
    """
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)

def directional_accuracy(df, entity_col, time_col, y_col, yhat_col):
    """
    Sign accuracy of month-on-month changes, averaged across LAs.
    """
    acc_list = []
    for la, sub in df.groupby(entity_col):
        sub = sub.sort_values(time_col)
        dy_true = sub[y_col].diff()
        dy_pred = sub[yhat_col].diff()
        mask = dy_true.notna() & dy_pred.notna() & (dy_true != 0)
        if mask.sum() == 0:
            continue
        correct = np.sign(dy_true[mask]) == np.sign(dy_pred[mask])
        acc_list.append(correct.mean())
    if not acc_list:
        return np.nan
    return float(np.mean(acc_list))

def morans_i(residuals, xs, ys, k=5):
    """
    Moran's I using k-NN weights on coordinates.
    residuals: [N], xs, ys: [N]
    """
    residuals = np.asarray(residuals)
    xs = np.asarray(xs)
    ys = np.asarray(ys)
    N = len(residuals)
    coords = np.column_stack([xs, ys])

    nbrs = NearestNeighbors(n_neighbors=k+1).fit(coords)
    _, indices = nbrs.kneighbors(coords)

    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        for j in indices[i, 1:]:
            W[i, j] = 1.0
            W[j, i] = 1.0

    S0 = W.sum()
    if S0 == 0:
        return np.nan

    x = residuals
    x_bar = x.mean()
    num = 0.0
    for i in range(N):
        for j in range(N):
            num += W[i, j] * (x[i] - x_bar) * (x[j] - x_bar)
    den = ((x - x_bar) ** 2).sum()
    if den == 0:
        return np.nan

    I = (N / S0) * (num / den)
    return float(I)

## Load data and centroids

In [ ]:
df_full = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df_full = df_full.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

# robust centroids (for Moran's I later)
df_full[["centroid_x", "centroid_y"]] = (
    df_full.groupby(ENTITY_COL)[["centroid_x", "centroid_y"]]
           .ffill()
           .bfill()
)

centroid_df = (
    df_full.drop_duplicates(ENTITY_COL)
           .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

# main df: from 2007-04 onwards
df = df_full[df_full[TIME_COL] >= pd.Timestamp("2007-04-01")].copy()
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)
print("Number of LAs used:", N)

centroid_df = centroid_df.loc[la_order]
bad_las = centroid_df[centroid_df.isna().any(axis=1)].index.tolist()
if bad_las:
    print(f"⚠ Dropping {len(bad_las)} LAs with missing centroids:", bad_las)
    centroid_df = centroid_df.dropna()
    la_order = centroid_df.index.tolist()
    df = df[df[ENTITY_COL].isin(la_order)].copy()
    N = len(la_order)
    print("Updated number of LAs:", N)


## Complete panel [T, M] and add price lags

In [ ]:
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
print("Total time steps:", T_total)

full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

feature_cols = base_feature_cols.copy()

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# ffill/bfill features + target within each LA
df_panel[feature_cols + [TARGET_COL]] = (
    df_panel[feature_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

# ---- add price lags 1 and 12 ----
df_panel["price_lag1"] = (
    df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(1)
)
df_panel["price_lag12"] = (
    df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(12)
)

df_panel[["price_lag1", "price_lag12"]] = (
    df_panel[["price_lag1", "price_lag12"]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

lag_price_cols = ["price_lag1", "price_lag12"]
feature_cols = feature_cols + lag_price_cols

# last-resort fill
missing_total = df_panel[feature_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after lag creation. Filling with column means.")
    col_means = df_panel[feature_cols + [TARGET_COL]].mean()
    df_panel[feature_cols + [TARGET_COL]] = df_panel[feature_cols + [TARGET_COL]].fillna(col_means)

print("NaNs after panel completion:",
      df_panel[feature_cols + [TARGET_COL]].isna().sum().sum())

F = len(feature_cols)

X_all = (
    df_panel[feature_cols]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N, F)
)
y_all = (
    df_panel[TARGET_COL]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N)
)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

y_all_orig = y_all.copy()

## Train val test time indices

In [ ]:
train_end_idx  = np.searchsorted(dates, TRAIN_END_DATE,  side="right")
val_end_idx    = np.searchsorted(dates, VAL_END_DATE,    side="right")
test_start_idx = np.searchsorted(dates, TEST_START_DATE, side="left")

print(f"Train ends at idx {train_end_idx-1}, date {dates[train_end_idx-1].date()}")
print(f"Val   ends at idx {val_end_idx-1}, date {dates[val_end_idx-1].date()}")
print(f"Test starts at idx {test_start_idx}, date {dates[test_start_idx].date()}")


## Scaling

In [ ]:
X_tv_flat = X_all[:val_end_idx].reshape(-1, F)
y_tv_flat = y_all[:val_end_idx].reshape(-1, 1)

x_scaler = StandardScaler()
X_all_scaled = X_all.copy()
X_all_scaled[:val_end_idx] = x_scaler.fit_transform(X_tv_flat).reshape(-1, N, F)
X_all_scaled[val_end_idx:] = x_scaler.transform(
    X_all[val_end_idx:].reshape(-1, F)
).reshape(-1, N, F)

y_scaler = RobustScaler()
y_all_scaled = y_all.copy()
y_all_scaled[:val_end_idx] = y_scaler.fit_transform(y_tv_flat).reshape(-1, N)
y_all_scaled[val_end_idx:] = y_scaler.transform(
    y_all[val_end_idx:].reshape(-1, 1)
).reshape(-1, N)

y_scale_factor = float(y_scaler.scale_[0])
print("NaNs in X_all_scaled:", np.isnan(X_all_scaled).sum())
print("NaNs in y_all_scaled:", np.isnan(y_all_scaled).sum())

## Dataset class

In [ ]:
class PanelWindowDataset(Dataset):
    """
    Each sample is (X_seq, y_t, t_idx, n_idx):
      - X_seq: [window, F]
      - y_t: scalar
      - t_idx: time index
      - n_idx: LA index
    """
    def __init__(self, X_all, y_all, window, t_start, t_end):
        self.X_all = X_all
        self.y_all = y_all
        self.window = window
        self.T, self.N, self.F = X_all.shape

        indices = []
        for t in range(t_start, t_end):
            if t - window < 0:
                continue
            for n in range(self.N):
                indices.append((t, n))
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t, n = self.indices[idx]
        X_seq = self.X_all[t - self.window:t, n, :]  # [window, F]
        y_t   = self.y_all[t, n]
        return (
            torch.tensor(X_seq, dtype=torch.float32),
            torch.tensor(y_t,   dtype=torch.float32),
            t,
            n
        )

train_ds = PanelWindowDataset(X_all_scaled, y_all_scaled,
                              window=WINDOW,
                              t_start=WINDOW,
                              t_end=train_end_idx)
val_ds   = PanelWindowDataset(X_all_scaled, y_all_scaled,
                              window=WINDOW,
                              t_start=train_end_idx,
                              t_end=val_end_idx)
test_ds  = PanelWindowDataset(X_all_scaled, y_all_scaled,
                              window=WINDOW,
                              t_start=test_start_idx,
                              t_end=T_total)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))
print("Test samples:", len(test_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


## CNN-LSTM model

In [ ]:
class CNNLSTM(nn.Module):
    def __init__(self, in_feats, cnn_channels, kernel_size, lstm_hidden, dropout=0.0):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=in_feats,
            out_channels=cnn_channels,
            kernel_size=kernel_size,
            padding="same"
        )
        self.lstm = nn.LSTM(
            input_size=cnn_channels,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden, 1)

    def forward(self, X_seq):
        """
        X_seq: [B, T, F]
        """
        B, T, F = X_seq.shape
        x = X_seq.permute(0, 2, 1)       # [B, F, T]
        x = self.conv(x)                 # [B, C_out, T]
        x = torch.relu(x)
        x = x.permute(0, 2, 1)           # [B, T, C_out]
        out, (h_n, c_n) = self.lstm(x)   # h_n: [1, B, H]
        h_last = h_n[-1]                 # [B, H]
        h_last = self.dropout(h_last)
        y_hat = self.fc(h_last).squeeze(-1)  # [B]
        return y_hat

## Train with early stopping

In [ ]:
model = CNNLSTM(
    in_feats=F,
    cnn_channels=CNN_CHANNELS,
    kernel_size=KERNEL_SIZE,
    lstm_hidden=LSTM_HIDDEN,
    dropout=DROPOUT
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()

best_val_mse = np.inf
best_epoch   = -1
epochs_no_improve = 0
best_state = None

print("\n=== Training final CNN-LSTM with early stopping ===")
start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_losses = []

    for X_seq, y_t, t_idx, n_idx in train_loader:
        X_seq = X_seq.to(DEVICE)  # [B, T, F]
        y_t   = y_t.to(DEVICE)    # [B]

        optimizer.zero_grad()
        y_hat = model(X_seq)
        loss  = loss_fn(y_hat, y_t)

        if not torch.isfinite(loss):
            print(f"  ⚠ Non-finite training loss at epoch {epoch}. Aborting training.")
            train_losses = []
            break

        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        train_losses.append(loss.item())

    if not train_losses:
        break

    # validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for X_seq, y_t, t_idx, n_idx in val_loader:
            X_seq = X_seq.to(DEVICE)
            y_t   = y_t.to(DEVICE)
            y_hat = model(X_seq)
            vloss = loss_fn(y_hat, y_t)
            if torch.isfinite(vloss):
                val_losses.append(vloss.item())

    if not val_losses:
        print("  ⚠ All val losses non-finite; stopping.")
        break

    val_mse = float(np.mean(val_losses))
    val_rmse_orig = np.sqrt(val_mse) * y_scale_factor

    print(f"Epoch {epoch:03d} | "
          f"train MSE={np.mean(train_losses):.4f} | "
          f"val MSE={val_mse:.4f} | "
          f"val RMSE(£)≈{val_rmse_orig:,.1f}")

    if val_mse + 1e-6 < best_val_mse:
        best_val_mse = val_mse
        best_epoch   = epoch
        epochs_no_improve = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

train_time = time.time() - start_time
print(f"Training finished in {train_time:.1f} seconds. Best epoch: {best_epoch}")

if best_state is not None:
    model.load_state_dict(best_state)
else:
    print("⚠ No best_state captured; using last epoch weights.")

## Test evaluation

In [ ]:
model.eval()
# we’ll fill arrays [T, N] for test period
y_pred_full_scaled = np.full_like(y_all_scaled, np.nan, dtype=np.float32)

with torch.no_grad():
    for X_seq, y_t, t_idx, n_idx in test_loader:
        X_seq = X_seq.to(DEVICE)
        y_hat = model(X_seq).cpu().numpy()  # [B]

        t_idx = np.array(t_idx)
        n_idx = np.array(n_idx)

        for i in range(len(t_idx)):
            t = t_idx[i]
            n = n_idx[i]
            y_pred_full_scaled[t, n] = y_hat[i]

# flatten test period arrays
test_mask = np.zeros((T_total, N), dtype=bool)
test_mask[test_start_idx:T_total, :] = True

y_true_test_orig = y_all_orig[test_mask]   # [S]
y_pred_test_scaled = y_pred_full_scaled[test_mask].reshape(-1, 1)

# inverse-transform predictions
y_pred_test_orig = y_scaler.inverse_transform(y_pred_test_scaled).ravel()

# build df_test with one row per (time, LA) for which we have predictions
t_coords, n_coords = np.where(test_mask)
dates_rep = dates[t_coords]
la_rep    = np.array(la_order)[n_coords]

df_test = pd.DataFrame({
    TIME_COL: dates_rep,
    ENTITY_COL: la_rep,
    "y_true": y_true_test_orig,
    "y_pred": y_pred_test_orig,
})
df_test["resid"] = df_test["y_true"] - df_test["y_pred"]

# For MASE scaling: use all pre-test y in original scale (train+val)
y_train_for_mase = y_all_orig[:val_end_idx].reshape(-1)


## Metrics

In [ ]:
# =========================================================
# GLOBAL METRICS
# =========================================================
global_mae   = mae(df_test["y_true"], df_test["y_pred"])
global_rmse  = rmse(df_test["y_true"], df_test["y_pred"])
global_smape = smape(df_test["y_true"], df_test["y_pred"])
global_mase  = mase(df_test["y_true"], df_test["y_pred"], y_train_for_mase, m=12)

print("\n=== Global accuracy ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:.3f}%")
print(f"MASE  : {global_mase:.3f}")

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae = (
    df_test.groupby(ENTITY_COL)
           .apply(lambda g: mae(g["y_true"], g["y_pred"]))
)

median_mae = float(np.median(la_mae.values))
p75_mae    = float(np.percentile(la_mae.values, 75))

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
la_resid_mean = df_test.groupby(ENTITY_COL)["resid"].mean()
centroids = centroid_df.loc[la_resid_mean.index][["centroid_x", "centroid_y"]]

I_moran = morans_i(
    residuals=la_resid_mean.values,
    xs=centroids["centroid_x"].values,
    ys=centroids["centroid_y"].values,
    k=5
)

monthly_resid = (
    df_test.groupby(TIME_COL)["resid"]
           .mean()
           .sort_index()
)

lb = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
lb_stat = float(lb["lb_stat"].iloc[0])
lb_p    = float(lb["lb_pvalue"].iloc[0])

print("\n=== Spatio-temporal diagnostics ===")
print(f"Moran's I (mean residuals across LAs): {I_moran:.4f}")
print(f"Ljung–Box Q(12): stat={lb_stat:.3f}, p={lb_p:.4f}")

# =========================================================
# DIRECTIONAL ACCURACY & GROWTH-RATE ERROR
# =========================================================
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, "y_true", "y_pred")

# approximate 12-month growth-rate error
y_pred_full_orig = np.full_like(y_all_orig, np.nan, dtype=np.float32)
for (t, n), yhat in zip(zip(t_coords, n_coords), y_pred_test_orig):
    y_pred_full_orig[t, n] = yhat

errs = []
for t in range(test_start_idx, T_total):
    t_prev = t - 12
    if t_prev < 0:
        continue
    true_t    = y_all_orig[t]      # [N]
    true_prev = y_all_orig[t_prev] # [N]
    pred_t    = y_pred_full_orig[t]# [N]
    mask = (~np.isnan(pred_t)) & (true_t > 0) & (true_prev > 0)
    if not mask.any():
        continue
    true_growth = np.log(true_t[mask]) - np.log(true_prev[mask])
    pred_growth = np.log(pred_t[mask]) - np.log(true_prev[mask])
    errs.append(np.abs(true_growth - pred_growth))

if errs:
    gre_mae = float(np.mean(np.concatenate(errs)))
else:
    gre_mae = np.nan

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")

In [ ]:

output_path = "../../results/cnn_lstm_final_test_results.csv"

summary_df = pd.DataFrame([{
    "model_type": "CNNLSTM",

    # --- hyperparameters ---
    "WINDOW": WINDOW,
    "CNN_CHANNELS": CNN_CHANNELS,
    "KERNEL_SIZE": KERNEL_SIZE,
    "LSTM_HIDDEN": LSTM_HIDDEN,
    "DROPOUT": DROPOUT,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,

    # --- global accuracy ---
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,

    # --- across-LA consistency ---
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,

    # --- spatio-temporal diagnostics ---
    "Morans_I": I_moran,
    "LjungBox_Q12": lb_stat,
    "LjungBox_p": lb_p,

    # --- optional / directional ---
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae,

    # --- efficiency ---
    "Training_Time_sec": train_time,
    "Test_Start_Date": TEST_START_DATE.strftime("%Y-%m-%d")
}])

summary_df.to_csv(output_path, index=False)

print("\n✅ Final CNN-LSTM evaluation metrics saved to:")
print(output_path)



